In [ ]:
# | default_exp features.fsd_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class FSDGenomewideEvaluator(FeatureEvaluator):
    """Extracts normalized densities for genomewide fragment size buckets.

    Derived metrics:
    - Bimodality index: mono-nucleosomal peak / di-nucleosomal valley
    - Shannon entropy of the size distribution
    - 143/166 ratio (classic cfDNA short-fragment proxy)
    - Per-chromosome 143/166 ratio (if chrom/region column available)
    """

    name = "FsdGenomewide"
    source_file = ".FSD.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            cols = set(df.columns)
            prefix = "fsd_gw"

            if "total" not in cols:
                log.warning("fsd_gw_missing_total_col")
                return extracted

            total_reads = df["total"].sum()
            if total_reads == 0:
                return extracted

            skip_cols = {"region", "total", "chrom"}
            for c in df.columns:
                if c not in skip_cols:
                    bucket_sum = df[c].sum()
                    extracted[f"{prefix}_density_{c}"] = float(bucket_sum / total_reads)

            if "140-144" in cols and "165-169" in cols:
                v143 = df["140-144"].sum()
                v166 = df["165-169"].sum()
                if v166 > 0:
                    extracted[f"{prefix}_143_166_ratio"] = float(v143 / v166)

            # --- Derived: bimodality index ---
            peak_bins = ["140-144", "145-149", "150-154", "155-159"]
            valley_bins = [
                "170-174",
                "175-179",
                "180-184",
                "185-189",
                "190-194",
                "195-199",
            ]
            peak_sum = sum(df[b].sum() for b in peak_bins if b in cols)
            valley_sum = sum(df[b].sum() for b in valley_bins if b in cols)
            if valley_sum > 0:
                extracted[f"{prefix}_bimodality_index"] = float(peak_sum / valley_sum)

            # --- Derived: Shannon entropy ---
            densities = [v for k, v in extracted.items() if f"{prefix}_density_" in k]
            if densities:
                arr = np.array(densities)
                pos = arr[arr > 0]
                if len(pos) > 0:
                    extracted[f"{prefix}_entropy"] = float(-np.sum(pos * np.log2(pos)))

            # --- Per-chromosome 143/166 ratio ---
            chrom_col = (
                "chrom" if "chrom" in cols else ("region" if "region" in cols else None)
            )
            if chrom_col and "140-144" in cols and "165-169" in cols:
                for chrom, grp in df.groupby(chrom_col):
                    ch = str(chrom).replace(" ", "_")
                    v166_chr = grp["165-169"].sum()
                    if v166_chr > 0:
                        extracted[f"{prefix}_{ch}_143_166_ratio"] = float(
                            grp["140-144"].sum() / v166_chr
                        )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSDEvaluator()
df_test = pd.DataFrame({"140-144": [200], "165-169": [500], "total": [2000]})
res = evaluator.extract(df_test)
assert "fsd_gw_density_140-144" in res
assert res["fsd_gw_density_140-144"] == 0.1
assert res["fsd_gw_143_166_ratio"] == 0.4